# HPO-Läufe auf Google Colab

Führt die Hyperparameter-Optimierung (Optuna) auf einer Colab-GPU aus.
Abbruchsicher: Studies werden alle 10 Minuten nach Google Drive gesichert
und beim nächsten Start automatisch fortgesetzt.

**Vorbereitung (einmalig, siehe `REMOTE.md`):**
1. Colab Pro (für Background Execution — Laptop darf zu)
2. Secret `GITHUB_TOKEN` hinterlegen (Schlüssel-Symbol links)
3. Laufzeit → Laufzeittyp ändern → **GPU (T4)**

Dann: Lauf-Parameter in der Zelle unten setzen → *Laufzeit → Alle ausführen*.

In [ ]:
# GPU prüfen und Google Drive verbinden (Sicherungsort)
!nvidia-smi -L

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/thesis_hpo')
(DRIVE_DIR / 'studies').mkdir(parents=True, exist_ok=True)
print('Sicherungsort:', DRIVE_DIR)

In [ ]:
# Repo klonen (privates Repo -> Token aus Colab-Secrets)
from google.colab import userdata

REPO = 'GITHUB_USER/masterthesis-code'   # <-- ANPASSEN nach Repo-Erstellung

token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/thesis
!git clone --depth 1 https://{token}@github.com/{REPO}.git /content/thesis
%cd /content/thesis
!git log --oneline -1

In [ ]:
# Abhängigkeiten installieren (torch bleibt die Colab-CUDA-Version)
%pip install -q -r requirements.txt
import torch, darts
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| darts', darts.__version__)

In [ ]:
# Resume: vorhandene Optuna-Studies von Drive zurueckholen
import shutil
from pathlib import Path

studies_dir = Path('output/optimization/studies')
studies_dir.mkdir(parents=True, exist_ok=True)

restored = 0
for db in (DRIVE_DIR / 'studies').glob('*.db'):
    shutil.copy2(db, studies_dir / db.name)
    restored += 1
    print('wiederhergestellt:', db.name)
print(f'{restored} Study-Datenbank(en) von Drive uebernommen.' if restored else 'Kein Resume-Stand vorhanden — frischer Start.')

In [ ]:
# ========== HPO-Lauf ==========
MODEL   = 'xgboost'      # 'xgboost' | 'tft' | 'regression'
HORIZON = 'day_ahead'    # 'day_ahead' | 'week_ahead'
TIMEOUT = 6 * 3600       # Sekunden (6 h); Optuna startet danach keinen neuen Trial

import subprocess, time, shutil
from pathlib import Path

cmd = (f'python -u main.py optimize --model {MODEL} --target residual_load '
       f'--horizon {HORIZON} --timeout {TIMEOUT}')
print('Starte:', cmd, flush=True)
proc = subprocess.Popen(cmd, shell=True)

# Waehrend der Lauf arbeitet: alle 10 Minuten die Studies nach Drive sichern
while proc.poll() is None:
    time.sleep(600)
    for db in Path('output/optimization/studies').glob('*.db'):
        shutil.copy2(db, DRIVE_DIR / 'studies' / db.name)
    print('Zwischenstand gesichert', time.strftime('%H:%M'), flush=True)

print('Lauf beendet, Exit-Code:', proc.returncode)

In [ ]:
# Endstand sichern: Studies, beste Parameter, Trial-Tabellen -> Drive
import shutil
from pathlib import Path

opt_dir = Path('output/optimization')
for db in (opt_dir / 'studies').glob('*.db'):
    shutil.copy2(db, DRIVE_DIR / 'studies' / db.name)
for pattern in ('*_best_params.json', '*_trials.csv'):
    for f in opt_dir.glob(pattern):
        shutil.copy2(f, DRIVE_DIR / f.name)

print('Gesichert nach', DRIVE_DIR, ':')
for f in sorted(DRIVE_DIR.rglob('*')):
    if f.is_file():
        print(' ', f.relative_to(DRIVE_DIR), f'({f.stat().st_size/1024:.0f} KB)')

## Ergebnisse zurück auf den Laptop

Alles liegt in **Google Drive → `MyDrive/thesis_hpo/`**:

- `studies/*.db` → lokal nach `output/optimization/studies/`
- `*_best_params.json`, `*_trials.csv` → lokal nach `output/optimization/`

Danach lokal das finale Training mit den optimierten Parametern starten.
Für den nächsten Lauf: oben `MODEL`/`HORIZON` ändern und erneut *Alle ausführen*.